# Imports and set-up

In [1]:
import ase.io

import sys
import os
import glob
import subprocess
import json

In [2]:

current_folder = os.getcwd()

# 1. Get the path to the directory two levels up
parent = os.path.abspath(os.path.join(current_folder, ".."))

# 2. Build the full path to the internal folder
target_dir = os.path.join(parent, "tabgap_install", "tabgap", "tabgap")

# 3. Add to sys.path
if target_dir not in sys.path:
    sys.path.append(target_dir)

# 4. Imports
from gap_fit import *
from gap_fit_params_default import *

In [3]:
# Multiple species
db = ase.io.read('../jgap_install/build/resources/xyz-samples/feni-train.xyz', index=':')

# Validate regression coefficients

### Use tabgap's default descriptor params, but smaller

In [4]:
if 'soap' in descriptors:
    del descriptors['soap']

descriptors['distance_2b']['n_sparse'] = 5
descriptors['eam_density']['n_sparse'] = 10
descriptors['angle_3b']['n_sparse'] = 10

if 'distance_2b' in descriptors:
    #del descriptors['distance_2b']
    pass

if 'eam_density' in descriptors:
    del descriptors['eam_density']
    pass
if 'angle_3b' in descriptors:
    del descriptors['angle_3b']
    pass
descriptors_vec = [v for k, v in descriptors.items()]
descriptors_vec


[{'name': 'distance_2b',
  'cutoff': '5.0',
  'cutoff_transition_width': '1.0',
  'covariance_type': 'ard_se',
  'delta': '10.0',
  'theta_uniform': '1.0',
  'sparse_method': 'uniform',
  'n_sparse': 5,
  'print_sparse_index': 'sparse_indices_2b.out',
  'add_species': 'T'}]

In [5]:
# ? 'config_type_sigma': '{isolated_atom:0.0001:0.04:0.01:0.0:liquid:0.01:0.5:2.0:0.0:liquid_composition:0.01:0.5:2.0:0.0:liquid_hea:0.01:0.5:2.0:0.0:surf_liquid:0.01:0.4:0.2:0.0:dimer:0.1:1.0:1.0:0.0:short_range:0.05:0.8:0.8:0.0:hea_short_range:0.05:0.8:2.0:0.0:hea_small:0.01:0.1:0.5:0.0:composition:0.01:0.1:0.5:0.0:binary_alloys:0.01:0.1:0.5:0.0:hea_vacancies:0.01:0.1:0.5:0.0:hea_ints:0.01:0.1:0.5:0.0:hea_vac_saddle:0.01:0.1:0.5:0.0}',

global_args

{'sparse_jitter': '1e-8',
 'do_copy_at_file': 'False',
 'gp_file': 'gap.xml',
 'rnd_seed': '999',
 'default_sigma': '{0.002 0.1 0.2 0.5}'}

## Run QUIP GAP fit
### Setup env variable used by tabgap util

In [6]:
# Your base path
base_build_dir = os.path.join(parent, "quip_install", "QUIP", "build")

# Use glob to find the directory inside (the * matches the one dir)
# Result is a list, so we take the first element [0]
target_dir = glob.glob(os.path.join(base_build_dir, "*"))[0]

# Now join 'quip' to that resolved path
final_path = os.path.join(target_dir, "gap_fit")

print(final_path)

os.environ.setdefault('GAP_FIT', final_path)

/Users/jegorsbalzins/jgap/benchmarking/quip_install/QUIP/build/darwin_x86_64_gfortran/gap_fit


'/Users/jegorsbalzins/jgap/benchmarking/quip_install/QUIP/build/darwin_x86_64_gfortran/gap_fit'

### Run

In [7]:
for i in range(len(db)):
    a = db[i]
    if 'trimer' in a.info['config_type']:
        #print(i)
        #print(a)
        pass
    if 'virial' in a.info:
        #del a.info['virial']
        pass

to_be_fit = [db[0], db[1], db[2]]
#del to_be_fit[0].arrays['force']
#del to_be_fit[1].arrays['force']

run(to_be_fit, global_args, descriptors_vec, compute_errors=False, rundir='quip-out')

## Fit JGAP with same sparse points

In [8]:
os.chdir(current_folder + "/quip-out")
subprocess.run('../../jgap_install/build/jgap_convert_quip_xml_app gap.xml', shell=True)
os.chdir('..')

2026-04-26 23:22:56.964932 [INFO ] jGAP from QUIP xml v5.0.0
2026-04-26 23:22:56.966390 [INFO ] Converting QUIP GAP to jGAP: gap.xml
2026-04-26 23:22:56.966543 [INFO ] Converting
2026-04-26 23:22:56.966701 [INFO ] Converted => saving to: gap.xml.jgap.json


In [9]:
with open('quip-out/gap.xml.jgap.json') as quip_gap_converted:
    sparse_pts_with_coeffs = json.load(quip_gap_converted)

In [10]:
print(sparse_pts_with_coeffs['potentials']['GAP']['type'])

descriptors_without_coeffs = sparse_pts_with_coeffs['potentials']['GAP']['descriptors']
for label, desc in descriptors_without_coeffs.items():
    for kernel in desc:
        if 'coefficient' in kernel:
            del kernel['coefficient']

print(descriptors_without_coeffs)

gap
{'0': {'cutoff': {'cutoff': 5.0, 'cutoff_transition_width': 1.0, 'type': 'coscutoff'}, 'kernels': [{'coefficient': 8.328209275640678, 'descriptor_prefactors': 0.7576632909575536, 'energy_scale': 10.0, 'length_scale': 1.0, 'r': 4.327670629591253, 'species_pair': ['Ni', 'Ni'], 'type': 'squared_exp'}, {'coefficient': -0.3091518959294796, 'descriptor_prefactors': 1.0, 'energy_scale': 10.0, 'length_scale': 1.0, 'r': 3.6916919226009273, 'species_pair': ['Ni', 'Ni'], 'type': 'squared_exp'}, {'coefficient': 0.01777499716039789, 'descriptor_prefactors': 1.0, 'energy_scale': 10.0, 'length_scale': 1.0, 'r': 2.603004671285298, 'species_pair': ['Ni', 'Ni'], 'type': 'squared_exp'}, {'coefficient': 8.641937487463146, 'descriptor_prefactors': 0.3440111805812503, 'energy_scale': 10.0, 'length_scale': 1.0, 'r': 4.600991403773756, 'species_pair': ['Ni', 'Ni'], 'type': 'squared_exp'}, {'coefficient': -14.825917775196546, 'descriptor_prefactors': 0.6052725514928905, 'energy_scale': 10.0, 'length_scale'

In [11]:
jgap_params = {
    "output_file": "jgap-coeffs.jgap.json",
    "training_data_xyz": "../quip-out/db_train.xyz",
    "fit_order": ["isolated_atom", "GAP"],
    "use_virials": True,
    "fits": {
        "isolated_atom": {
            "type": "isolated_atom"
        },
        "GAP": {
            "type": "qr_gap",
            "regularization_rules": {
                "type": "per_ct",
                "E": 0.002,
                "F": 0.1,
                "V": 0.2,
                "per_keyword": [
                    # { "contains": "isolated_atom", "multiplier": 0.1 }
                ]
            },
            "descriptors": descriptors_without_coeffs
        }
    }
}
os.makedirs("jgap-out", exist_ok=True)
with open('jgap-out/fit-params.json', 'w') as jgap:
    json.dump(jgap_params, jgap)

In [12]:
os.chdir(current_folder + "/jgap-out")
subprocess.run('../../jgap_install/build/jgap_fit_app fit-params.json', shell=True)
os.chdir('..')

2026-04-26 23:22:57.028248 [INFO ] jGAP fit v5.0.0
2026-04-26 23:22:57.031030 [INFO ] Output file name: jgap-coeffs.jgap.json
2026-04-26 23:22:57.031042 [INFO ] Picking fit logic
2026-04-26 23:22:57.031062 [INFO ] Picking fitting logic for GAP
2026-04-26 23:22:57.031082 [DEBUG] Parsing 2b descriptor params
2026-04-26 23:22:57.031126 [INFO ] Picking fitting logic for isolated_atom
2026-04-26 23:22:57.031141 [INFO ] Reading training data
2026-04-26 23:22:57.032212 [INFO ] Fitting "composite" potential with params from file fit-params.json: {"fit_order":["isolated_atom","GAP"],"fits":{"GAP":{"descriptors":{"0":{"cutoff":{"cutoff":5.0,"cutoff_transition_width":1.0,"type":"coscutoff"},"kernels":[{"coefficient":8.328209275640678,"descriptor_prefactors":0.7576632909575536,"energy_scale":10.0,"length_scale":1.0,"r":4.327670629591253,"species_pair":["Ni","Ni"],"type":"squared_exp"},{"coefficient":-0.3091518959294796,"descriptor_prefactors":1.0,"energy_scale":10.0,"length_scale":1.0,"r":3.691691